# Module 4 — FeatureCollections & Zonal Stats at Scale

**Goal:** go from "one point" to "thousands of polygons" (like your COMTRS chunks) efficiently, using `reduceRegions` instead of looping `reduceRegion` in Python.

**Time:** ~2 hours — this is the module most directly reusable in the glyphosate pipeline.

### Why looping is a trap
If you write a Python `for` loop that calls `.reduceRegion().getInfo()` once per polygon per image, you're doing one network round-trip per (polygon × image) pair. For a state-wide, year-long COMTRS run that's potentially millions of round trips — it will be unusably slow and will hit rate limits. The fix is to push the *looping* itself onto the server, using `reduceRegions` (plural) which reduces one image over **many** geometries in a single server-side operation, and `.map()` to iterate over images.


In [ ]:
import ee
try:
    ee.Initialize(project='YOUR-GCP-PROJECT-ID')
except Exception:
    ee.Authenticate()
    ee.Initialize(project='YOUR-GCP-PROJECT-ID')

import pandas as pd


### Uploading your own polygons: `ee.FeatureCollection`

Your COMTRS shapefile needs to become an `ee.FeatureCollection`. For a one-off/small file, you can upload it as a GEE Asset via the code editor or `earthengine upload table`, then load it by asset ID. For prototyping, you can also build a small `FeatureCollection` directly in Python from a handful of geometries — that's what we'll do here so this notebook runs standalone.

In [ ]:
# Toy stand-in for a few COMTRS-like chunks (~1 sq mile boxes) in the Central Valley.
# In the real pipeline you'd instead do:
#   comtrs = ee.FeatureCollection('projects/YOUR-PROJECT/assets/comtrs_california')
def make_box(lon, lat, size_deg=0.014):
    return ee.Geometry.Rectangle([lon - size_deg, lat - size_deg, lon + size_deg, lat + size_deg])

chunks = ee.FeatureCollection([
    ee.Feature(make_box(-121.5, 38.5), {'CO_MTRS': 'CHUNK_A'}),
    ee.Feature(make_box(-121.3, 38.6), {'CO_MTRS': 'CHUNK_B'}),
    ee.Feature(make_box(-121.7, 38.4), {'CO_MTRS': 'CHUNK_C'}),
])
print(chunks.size().getInfo())


### `reduceRegions` — one image, many polygons, one call

In [ ]:
def mask_s2_clouds(image):
    qa = image.select('QA60')
    mask = qa.bitwiseAnd(1 << 10).eq(0).And(qa.bitwiseAnd(1 << 11).eq(0))
    return image.updateMask(mask)

img = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
       .filterDate('2021-07-01', '2021-07-15')
       .filterBounds(chunks)
       .map(mask_s2_clouds)
       .median())

ndvi = img.normalizedDifference(['B8', 'B4']).rename('NDVI')

stats = ndvi.reduceRegions(collection=chunks, reducer=ee.Reducer.mean(), scale=10)
print(stats.getInfo())


One call, three polygons, mean NDVI for each — no Python loop involved. This scales to thousands of polygons the same way (subject to GEE's per-request size limits, which is why large jobs get batched — more on that below).

### Exercise 4.1
Add `ee.Reducer.count()` combined with `.mean()` (via `.combine(reducer2=ee.Reducer.count(), sharedInputs=True)`) so you get both the mean NDVI and the number of valid (unmasked) pixels per chunk in the same call. This lets you flag/drop chunks with too few clear pixels — important for chunks that are mostly cloud-covered on a given date.


In [ ]:
# TODO: your code here


### Doing this across a whole year: `map` over images, `reduceRegions` inside

Now combine what you learned in Module 2 with `reduceRegions`: map over a year of images, and inside each image's mapped function, reduce over all your chunks at once. Each mapped call returns a `FeatureCollection`; `.flatten()` on the resulting `ImageCollection`-of-`FeatureCollections` merges them into one long table.

In [ ]:
def reduce_one_image(image):
    ndvi = image.normalizedDifference(['B8', 'B4']).rename('NDVI')
    date_str = image.date().format('YYYY-MM-dd')
    reduced = ndvi.reduceRegions(collection=chunks, reducer=ee.Reducer.mean(), scale=10)
    return reduced.map(lambda f: f.set('date', date_str))

collection = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
              .filterDate('2021-06-01', '2021-08-31')
              .filterBounds(chunks)
              .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 60))
              .map(mask_s2_clouds))

long_table = collection.map(reduce_one_image).flatten()
records = long_table.getInfo()['features']
df = pd.DataFrame([r['properties'] for r in records])
df = df.dropna(subset=['NDVI'])
df.pivot_table(index='date', columns='CO_MTRS', values='NDVI')


That pivoted table — dates as rows, chunk IDs as columns, NDVI as values — is essentially the raw input the glyphosate model needs before you compute per-chunk baselines and look for anomalies.

### On scaling to the real, large job
`.getInfo()` on a huge `FeatureCollection` will eventually time out or hit size limits. For a genuinely state-wide, year-long run over real COMTRS polygons, the production move is `ee.batch.Export.table.toDrive()` (or `toCloudStorage`), which runs asynchronously on Google's servers and writes a CSV you download afterward instead of streaming through `.getInfo()`. We'll cover exports properly in Module 5.

### Project 4 — "Zonal time series extractor"

Write a function:

```python
def extract_zonal_timeseries(feature_collection, id_field, start, end, index='NDVI',
                              collection_id='COPERNICUS/S2_SR_HARMONIZED', scale=10) -> pd.DataFrame
```

that returns a long-format DataFrame (`date`, `id_field`, index value, pixel count) for **every polygon** in the given `FeatureCollection` over the date range, using the `map` + `reduceRegions` + `flatten` pattern above (not a Python loop). Test it against the toy `chunks` collection, then think through (in a markdown cell, no need to run it) how you'd point this at your real ~39MB COMTRS asset for the full state and a full year — what would you batch by (date range chunks? spatial tiles?) to stay under request limits.


In [ ]:
def extract_zonal_timeseries(feature_collection, id_field, start, end, index='NDVI',
                              collection_id='COPERNICUS/S2_SR_HARMONIZED', scale=10):
    # TODO
    pass


---
**Checkpoint:** `reduceRegions` + `map` + `flatten` should now feel like your default pattern for "extract a stat per polygon per image, at scale" — never a Python loop calling `reduceRegion` one polygon at a time. Next module: exporting really large jobs asynchronously, and building the harmonic regression baseline your change-detection logic needs.